# RFdiffusion & ProteinMPNN Tutorial

This notebook teaches you the basics of:
- **RFdiffusion**: Generate protein backbone structures
- **ProteinMPNN**: Design sequences for protein structures

## What are these tools?

- **RFdiffusion**: Uses diffusion models to generate new protein structures (backbones only)
- **ProteinMPNN**: Designs amino acid sequences that fold into given protein structures

Together, they enable: Structure Generation → Sequence Design → Full Protein Design


## Part 1: Setup


In [ ]:
# Install dependencies
import os
import sys
import subprocess

# Check if virtual environment exists and activate it
venv_path = os.path.join(os.getcwd(), "venv")
if os.path.exists(venv_path):
    venv_python = os.path.join(venv_path, "bin", "python3")
    if os.path.exists(venv_python):
        # Update sys.executable to use venv Python
        sys.executable = venv_python
        # Add venv to path
        venv_site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
        if venv_site_packages not in sys.path:
            sys.path.insert(0, venv_site_packages)
        print(f"✓ Using virtual environment: {venv_path}")
    else:
        print("⚠️  Venv directory exists but python3 not found, using system Python")
else:
    print("⚠️  No virtual environment found, using system Python")
    print("   Consider creating one with: python3 -m venv venv")

# Check if RFdiffusion exists
if not os.path.isdir("RFdiffusion"):
    print("Cloning RFdiffusion...")
    os.system("git clone https://github.com/sokrypton/RFdiffusion.git")
else:
    print("RFdiffusion already exists")

# Download model checkpoints if needed
models_dir = "RFdiffusion/models"
base_ckpt = os.path.join(models_dir, "Base_ckpt.pt")

if not os.path.exists(base_ckpt):
    print("\nDownloading RFdiffusion model checkpoints (this may take a few minutes)...")
    os.makedirs(models_dir, exist_ok=True)
    
    models_to_download = {
        "Base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt",
        "Complex_base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",
        "Complex_Fold_base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/60f09a193fb5e5ccdc4980417708dbab/Complex_Fold_base_ckpt.pt"
    }
    
    for model_name, url in models_to_download.items():
        model_path = os.path.join(models_dir, model_name)
        if not os.path.exists(model_path):
            print(f"  Downloading {model_name}...")
            result = os.system(f"wget -q {url} -O {model_path}")
            if result == 0:
                print(f"  ✓ {model_name} downloaded")
            else:
                print(f"  ⚠️  Failed to download {model_name}")
    print("✓ Model checkpoints ready")
else:
    print("✓ RFdiffusion model checkpoints already exist")

# Add to path
if 'RFdiffusion' not in sys.path:
    sys.path.append('RFdiffusion')
    os.environ["DGLBACKEND"] = "pytorch"

# Install required packages using uv (faster and handles dependencies better)
# Check if uv is available (check common locations)
uv_available = False
uv_path = None

# Check common uv locations
uv_locations = ["uv", "/home/hansonwen/.local/bin/uv", os.path.expanduser("~/.local/bin/uv")]
for uv_cmd in uv_locations:
    try:
        result = subprocess.run([uv_cmd, "--version"], capture_output=True, text=True, timeout=5)
        if result.returncode == 0:
            uv_available = True
            uv_path = uv_cmd
            print(f"✓ Using uv: {result.stdout.strip()} (from {uv_cmd})")
            break
    except:
        continue

if not uv_available:
    print("⚠️  uv not found, falling back to pip")
    print("   Install uv with: curl -LsSf https://astral.sh/uv/install.sh | sh")

print("Installing basic dependencies and RFdiffusion requirements...")

# List of all required packages for RFdiffusion
base_packages = [
    "omegaconf", "hydra-core", "pyyaml",  # Configuration
    "opt_einsum", "e3nn==0.5.0",  # RFdiffusion core dependencies
    "pyrsistent", "icecream",  # Additional utilities
    "decorator==5.1.0"  # Specific version requirement
]

if uv_available:
    # Use uv pip install (installs all dependencies automatically)
    result = subprocess.run([uv_path, "pip", "install", "--python", sys.executable] + base_packages, 
                           capture_output=True, text=True, timeout=120)
else:
    # Fallback to pip
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + base_packages, 
                           capture_output=True, text=True, timeout=120)

if result.returncode == 0:
    print("✓ Basic dependencies and RFdiffusion requirements installed")
else:
    print(f"⚠️  Some dependencies may have failed: {result.stderr[:200]}")

# Install DGL (Deep Graph Library) - required for RFdiffusion
# Note: DGL installation depends on your CUDA version
# For CUDA 12.1: pip install dgl -f https://data.dgl.ai/wheels/cu121/repo.html
# For CPU only: pip install dgl
# Install DGL (Deep Graph Library) - required for RFdiffusion
print("Installing DGL...")

# Detect CUDA version
cuda_version = None
try:
    import subprocess
    # Try to get CUDA version from nvcc
    result = subprocess.run(["nvcc", "--version"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        for line in result.stdout.split('\n'):
            if 'release' in line.lower():
                import re
                match = re.search(r'release\s+(\d+\.\d+)', line)
                if match:
                    cuda_version = float(match.group(1))
                    print(f"Detected CUDA version: {cuda_version}")
                    break
except:
    pass

# Try to install CUDA version of DGL
dgl_installed = False
if cuda_version:
    # DGL supports cu118, cu121, cu124. For CUDA 13.0, try cu121 (backward compatible)
    cuda_major = int(cuda_version)
    if cuda_major >= 13:
        cuda_wheel = "cu121"  # CUDA 13.x is compatible with cu121
        print(f"CUDA {cuda_version} detected, using cu121 wheels (compatible)")
    elif cuda_major >= 12:
        if cuda_version >= 12.4:
            cuda_wheel = "cu124"
        else:
            cuda_wheel = "cu121"
    elif cuda_major >= 11:
        cuda_wheel = "cu118"
    else:
        cuda_wheel = None
    
    if cuda_wheel:
        print(f"Installing DGL with CUDA support ({cuda_wheel})...")
        try:
            # Try latest version first (2.1.0), then fallback to 2.0.0
            # Use uv for faster installation with all dependencies
            for dgl_version in ["2.1.0", "2.0.0"]:
                if uv_available:
                    # Use uv pip install (installs all dependencies automatically)
                    # Note: uv handles extra index URLs differently
                    result = subprocess.run(
                        [uv_path, "pip", "install", "--python", sys.executable,
                         f"dgl=={dgl_version}", 
                         "--extra-index-url", f"https://data.dgl.ai/wheels/{cuda_wheel}/repo.html"],
                        capture_output=True,
                        text=True,
                        timeout=120
                    )
                else:
                    # Fallback to pip (install all dependencies, not --no-dependencies)
                    pip_args = [sys.executable, "-m", "pip", "install"]
                    if not os.path.exists(os.path.join(os.getcwd(), "venv", "bin", "python3")):
                        pip_args.append("--break-system-packages")
                    pip_args.extend([f"dgl=={dgl_version}", 
                                    "-f", f"https://data.dgl.ai/wheels/{cuda_wheel}/repo.html"])
                    
                    result = subprocess.run(
                        pip_args,
                        capture_output=True,
                        text=True,
                        timeout=120
                    )
                # Check if installation succeeded (returncode 0 means success)
                if result.returncode == 0:
                    dgl_installed = True
                    print(f"✓ DGL CUDA version {dgl_version} installed successfully")
                    # Show any warnings but don't fail
                    if "warning" in result.stderr.lower() or "warning" in result.stdout.lower():
                        print("  (Note: Some warnings may appear, but installation succeeded)")
                    break
                else:
                    # Installation failed
                    error_msg = result.stderr[:300] if result.stderr else result.stdout[:300]
                    if dgl_version == "2.1.0":
                        print(f"Version {dgl_version} installation failed: {error_msg}")
                        print("Trying version 2.0.0...")
                    else:
                        print(f"CUDA installation failed: {error_msg}")
                        raise RuntimeError(f"Failed to install DGL with CUDA support: {error_msg}")
            
            if not dgl_installed:
                raise RuntimeError("Failed to install DGL with CUDA support")
        except Exception as e:
            print(f"CUDA installation error: {e}")
            raise RuntimeError(f"Failed to install DGL with CUDA support: {e}")
else:
    raise RuntimeError("CUDA not detected. Please install CUDA or use a system with CUDA support.")

# Install PyTorch and torchdata (required dependencies for DGL)
if dgl_installed:
    print("Installing PyTorch and torchdata (required for DGL)...")
    try:
        # Install PyTorch with compatible version (2.2.1 for DGL compatibility)
        # Install packages separately to avoid conflicts
        if uv_available:
            print("Installing PyTorch 2.2.1 (compatible with DGL)...")
            # Install torch 2.2.1 (latest version supported by DGL pre-built binaries)
            torch_result = subprocess.run(
                [uv_path, "pip", "install", "--python", sys.executable, "torch==2.2.1"],
                capture_output=True,
                text=True,
                timeout=300
            )
            if torch_result.returncode == 0:
                print("✓ PyTorch 2.2.1 installed")
        else:
            # Fallback to pip
            pip_args = [sys.executable, "-m", "pip", "install", "torch==2.2.1"]
            if not os.path.exists(os.path.join(os.getcwd(), "venv", "bin", "python3")):
                pip_args.append("--break-system-packages")
            
            torch_result = subprocess.run(
                pip_args,
                capture_output=True,
                text=True,
                timeout=300
            )
            if torch_result.returncode == 0:
                print("✓ PyTorch 2.2.1 installed successfully in venv")
            else:
                print(f"⚠️  PyTorch installation had issues: {torch_result.stderr[:200]}")
        
        # Install DGL dependencies (pandas, numpy<2, scipy, networkx, etc.)
        # Note: numpy must be < 2 for compatibility with PyTorch 2.2.1
        print("Installing DGL dependencies (pandas, numpy<2, scipy, networkx)...")
        if uv_available:
            dgl_deps_result = subprocess.run(
                [uv_path, "pip", "install", "--python", sys.executable, 
                 "pandas", "numpy<2", "scipy", "networkx", "requests", "tqdm", "psutil", "pydantic"],
                capture_output=True,
                text=True,
                timeout=120
            )
            if dgl_deps_result.returncode == 0:
                print("✓ DGL dependencies installed")
            else:
                print(f"⚠️  Some DGL dependencies had issues: {dgl_deps_result.stderr[:200]}")
        else:
            dgl_deps_result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "pandas", "numpy<2", "scipy", "networkx", "requests", "tqdm", "psutil", "pydantic"],
                capture_output=True,
                text=True,
                timeout=120
            )
            if dgl_deps_result.returncode == 0:
                print("✓ DGL dependencies installed")
        
        # Install torchdata 0.7.1 (compatible version for DGL 2.1.0)
        print("Installing torchdata 0.7.1 (compatible with DGL 2.1.0)...")
        if uv_available:
            torchdata_result = subprocess.run(
                [uv_path, "pip", "install", "--python", sys.executable, "torchdata==0.7.1"],
                capture_output=True,
                text=True,
                timeout=120
            )
        else:
            # Fallback to pip
            torchdata_args = [sys.executable, "-m", "pip", "install", "torchdata==0.7.1"]
            if not os.path.exists(os.path.join(os.getcwd(), "venv", "bin", "python3")):
                torchdata_args.append("--break-system-packages")
            
            torchdata_result = subprocess.run(
                torchdata_args,
                capture_output=True,
                text=True,
                timeout=120
            )
        if torchdata_result.returncode == 0:
            print("✓ torchdata 0.7.1 installed successfully")
        else:
            print(f"⚠️  torchdata installation had issues: {torchdata_result.stderr[:200]}")
        
        # Install SE3Transformer (required by RFdiffusion)
        print("Installing SE3Transformer...")
        if os.path.exists("RFdiffusion/env/SE3Transformer"):
            se3_result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-e", "RFdiffusion/env/SE3Transformer/"],
                capture_output=True,
                text=True,
                timeout=60
            )
            if se3_result.returncode == 0:
                print("✓ SE3Transformer installed")
            else:
                print(f"⚠️  SE3Transformer installation had issues: {se3_result.stderr[:200]}")
        else:
            print("⚠️  SE3Transformer directory not found, may need to clone RFdiffusion first")
    except Exception as e:
        print(f"⚠️  Dependency installation error: {e}")

# Verify DGL installation and check installation location
# Only show error if installation actually failed (dgl_installed is False)
if not dgl_installed:
    print("⚠️  Error: DGL installation failed. Please install manually:")
    if cuda_version:
        cuda_major = int(cuda_version)
        if cuda_major >= 13:
            cuda_wheel = "cu121"
        elif cuda_major >= 12:
            cuda_wheel = "cu124" if cuda_version >= 12.4 else "cu121"
        elif cuda_major >= 11:
            cuda_wheel = "cu118"
        else:
            cuda_wheel = "cu121"
        venv_pip = os.path.join(venv_path, "bin", "pip") if os.path.exists(venv_path) else "pip"
        print(f"   {venv_pip} install --no-dependencies dgl==2.1.0 -f https://data.dgl.ai/wheels/{cuda_wheel}/repo.html")
    else:
        print("   CUDA not detected. Cannot install CUDA version of DGL.")
else:
    # Installation succeeded, try to verify import
    print("\nVerifying installation...")
    try:
        import dgl
        print(f"✓ DGL installed successfully (version: {dgl.__version__})")
        
        # Check where DGL is installed
        dgl_path = dgl.__file__
        if "venv" in dgl_path:
            print(f"✓ DGL installed in venv: {dgl_path}")
            print("  → Confirmed: Installed in virtual environment")
        elif "site-packages" in dgl_path:
            print(f"✓ DGL installed in: {dgl_path}")
        else:
            print(f"✓ DGL installed at: {dgl_path}")
        
        # Check if CUDA backend is available
        try:
            import torch
            print(f"✓ PyTorch version: {torch.__version__}")
            if torch.cuda.is_available():
                print(f"✓ PyTorch CUDA available: {torch.version.cuda}")
            else:
                print("⚠️  PyTorch CUDA not available")
        except Exception as e:
            print(f"⚠️  Could not verify PyTorch: {e}")
        
        try:
            import dgl.backend as F
            if hasattr(F, 'cuda') and F.cuda.is_available():
                print("✓ DGL CUDA backend is available")
            else:
                print("⚠️  DGL installed but CUDA backend not available (using CPU)")
        except Exception as e:
            print(f"⚠️  Could not verify DGL CUDA backend: {e}")
    except ImportError as e:
        print(f"⚠️  DGL installed but import failed: {e}")
        print("   This may be due to missing dependencies.")
        print("   Missing dependency detected. Installing common DGL dependencies...")
        # Try to install missing dependencies
        if uv_available:
            fix_result = subprocess.run(
                [uv_path, "pip", "install", "--python", sys.executable, "pandas", "numpy", "scipy", "networkx"],
                capture_output=True,
                text=True,
                timeout=60
            )
        else:
            fix_result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "pandas", "numpy", "scipy", "networkx"],
                capture_output=True,
                text=True,
                timeout=60
            )
        if fix_result.returncode == 0:
            print("✓ Dependencies installed. Please restart the kernel and try again.")
        else:
            print("   Please install manually: pip install pandas numpy scipy networkx")
    except Exception as e:
        print(f"⚠️  Verification error: {e}")
        print("   DGL appears to be installed, but there may be dependency issues.")

print("\n" + "="*60)
print("Setup complete! All dependencies installed in virtual environment.")
print("="*60)


✓ Using virtual environment: /home/hansonwen/igem/venv
RFdiffusion already exists
✓ Using uv: uv 0.9.9 (from uv)
Installing basic dependencies and RFdiffusion requirements...
✓ Basic dependencies and RFdiffusion requirements installed
Installing DGL...
Detected CUDA version: 13.0
CUDA 13.0 detected, using cu121 wheels (compatible)
Installing DGL with CUDA support (cu121)...
✓ DGL CUDA version 2.1.0 installed successfully
Installing PyTorch and torchdata (required for DGL)...
Installing PyTorch 2.2.1 (compatible with DGL)...
✓ PyTorch 2.2.1 installed
Installing DGL dependencies (pandas, numpy<2, scipy, networkx)...
✓ DGL dependencies installed
Installing torchdata 0.7.1 (compatible with DGL 2.1.0)...
✓ torchdata 0.7.1 installed successfully
Installing SE3Transformer...
✓ SE3Transformer installed

Verifying installation...
⚠️  DGL installed but import failed: No module named 'torchdata.datapipes.iter.util'; 'torchdata.datapipes.iter' is not a package
   This may be due to missing depende

## Part 2: RFdiffusion Basics

RFdiffusion can generate protein structures in several modes:

### 1. Unconditional Generation
Generate a protein structure from scratch (no constraints)


In [10]:
# Example 1: Generate a simple 100-residue protein
# This creates a protein backbone with no constraints

command = """
python RFdiffusion/run_inference.py \\
    inference.output_prefix=outputs/unconditional_100 \\
    inference.num_designs=1 \\
    'contigmap.contigs=[100]' \\
    diffuser.T=50
"""

print("Command to run unconditional generation:")
print(command)
print("\nThis will create: outputs/unconditional_100_0.pdb")


Command to run unconditional generation:

python RFdiffusion/run_inference.py \
    inference.output_prefix=outputs/unconditional_100 \
    inference.num_designs=1 \
    'contigmap.contigs=[100]' \
    diffuser.T=50


This will create: outputs/unconditional_100_0.pdb


In [8]:
os.system(command)


/home/hansonwen/igem/RFdiffusion/run_inference.py:55: SyntaxWarning: invalid escape sequence '\d'
  m = re.match(".*_(\d+)\.pdb$", e)


[2025-11-20 12:10:33,407][inference.model_runners][INFO] - Reading checkpoint from /home/hansonwen/igem/RFdiffusion/inference/../models/Base_ckpt.pt
This is inf_conf.ckpt_path
/home/hansonwen/igem/RFdiffusion/inference/../models/Base_ckpt.pt


Error executing job with overrides: ['inference.output_prefix=outputs/unconditional_100', 'inference.num_designs=1', 'contigmap.contigs=[100]', 'diffuser.T=50']
Traceback (most recent call last):
  File "/home/hansonwen/igem/RFdiffusion/run_inference.py", line 46, in main
    sampler = iu.sampler_selector(conf)
              ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hansonwen/igem/RFdiffusion/inference/utils.py", line 526, in sampler_selector
    sampler = model_runners.SelfConditioning(conf)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/hansonwen/igem/RFdiffusion/inference/model_runners.py", line 43, in __init__
    self.initialize(conf)
  File "/home/hansonwen/igem/RFdiffusion/inference/model_runners.py", line 102, in initialize
    self.load_checkpoint()
  File "/home/hansonwen/igem/RFdiffusion/inference/model_runners.py", line 171, in load_checkpoint
    self.ckpt  = torch.load(
                 ^^^^^^^^^^^
  File "/home/hansonwen/igem/venv/lib/python3.12/site-pack

256

### 2. Motif Scaffolding
Generate a structure that includes a specific motif (e.g., active site)


In [ ]:
# Example 2: Scaffold around a motif
# Format: [length_before/motif_residues/length_after]
# This keeps the motif fixed and generates scaffold around it

command = """
python RFdiffusion/run_inference.py \\
    inference.output_prefix=outputs/motif_scaffold \\
    inference.num_designs=1 \\
    inference.input_pdb=your_motif.pdb \\
    'contigmap.contigs=[40/A10-20/40]' \\
    diffuser.T=50
"""

print("Command to scaffold around a motif:")
print(command)
print("\nExplanation:")
print("- 40: Generate 40 residues before the motif")
print("- A10-20: Keep residues 10-20 from chain A fixed")
print("- 40: Generate 40 residues after the motif")


### 3. Understanding Contigs

Contigs define what to generate and what to keep fixed:

- **Numbers** (e.g., `100`): Generate this many residues
- **Chain:ResidueRange** (e.g., `A10-20`): Keep these residues fixed
- **Chain:Number** (e.g., `A:80`): Generate binder of length 80 to chain A
- **Separate with `/`**: Multiple segments
- **Separate with `:`**: Multiple chains/contigs


In [ ]:
# Contig examples
examples = {
    "Unconditional monomer": "[100]",
    "Unconditional dimer": "[100:100]",
    "Scaffold motif": "[40/A10-20/40]",
    "Binder": "[A:80]",
    "Complex scaffold": "[30/A5-15/50/A20-25/30]",
    "Variable length": "[50-100]"  # Random length between 50-100
}

print("Common contig patterns:\n")
for name, contig in examples.items():
    print(f"{name:25} → {contig}")


## Part 3: Running RFdiffusion (Simple Example)

Let's run a simple example step by step


In [ ]:
# Step 1: Create output directory
os.makedirs("outputs", exist_ok=True)

# Step 2: Run RFdiffusion
# This generates a 50-residue protein backbone

cmd = (
    "python RFdiffusion/run_inference.py "
    "inference.output_prefix=outputs/tutorial_50 "
    "inference.num_designs=1 "
    "'contigmap.contigs=[50]' "
    "diffuser.T=50"
)

print("Running RFdiffusion...")
print(f"Command: {cmd}\n")

# Uncomment to run:
# os.system(cmd)

print("After running, check: outputs/tutorial_50_0.pdb")
print("Note: The output will have all residues as GLY (glycine) because only backbone is designed")


## Part 4: ProteinMPNN Basics

ProteinMPNN designs sequences for protein structures.

**Key concept**: Given a backbone structure, ProteinMPNN predicts which amino acids should go at each position.


In [ ]:
# Example: Design sequence for a structure

mpnn_command = """
python colabdesign/rf/designability_test.py \\
    --pdb=outputs/tutorial_50_0.pdb \\
    --loc=outputs/tutorial_50 \\
    --num_seqs=8 \\
    --num_recycles=3
"""

print("ProteinMPNN command:")
print(mpnn_command)
print("\nParameters:")
print("- --pdb: Input structure file")
print("- --loc: Output directory")
print("- --num_seqs: Number of sequences to design")
print("- --num_recycles: AlphaFold recycles for validation")
print("\nOutput: mpnn_results.csv with designed sequences and scores")


## Part 5: Complete Workflow Example

Here's a complete example combining RFdiffusion + ProteinMPNN:

In [ ]:
# Complete workflow function

def simple_protein_design(output_name, length=50, num_seqs=4):
    """
    Simple function to design a protein:
    1. Generate backbone with RFdiffusion
    2. Design sequence with ProteinMPNN
    
    Args:
        output_name: Name for output files
        length: Length of protein to generate
        num_seqs: Number of sequences to design
    """
    
    # Step 1: RFdiffusion - Generate backbone
    print(f"Step 1: Generating {length}-residue backbone...")
    
    rf_cmd = (
        f"python RFdiffusion/run_inference.py "
        f"inference.output_prefix=outputs/{output_name} "
        f"inference.num_designs=1 "
        f"'contigmap.contigs=[{length}]' "
        f"diffuser.T=50"
    )
    
    print(f"Running: {rf_cmd}")
    # os.system(rf_cmd)  # Uncomment to run
    
    pdb_file = f"outputs/{output_name}_0.pdb"
    print(f"Expected output: {pdb_file}\n")
    
    # Step 2: ProteinMPNN - Design sequence
    print(f"Step 2: Designing {num_seqs} sequences...")
    
    mpnn_cmd = (
        f"python colabdesign/rf/designability_test.py "
        f"--pdb={pdb_file} "
        f"--loc=outputs/{output_name} "
        f"--num_seqs={num_seqs} "
        f"--num_recycles=3"
    )
    
    print(f"Running: {mpnn_cmd}")
    # os.system(mpnn_cmd)  # Uncomment to run
    
    print(f"\nResults will be in: outputs/{output_name}/")
    print(f"- mpnn_results.csv: Sequences and scores")
    print(f"- best.pdb: Best designed structure")

# Example usage
print("Complete workflow function:")
print("=" * 50)
simple_protein_design("my_protein", length=60, num_seqs=8)


## Part 6: Quick Reference

### RFdiffusion Command Template:

```bash
python RFdiffusion/run_inference.py \
    inference.output_prefix=outputs/NAME \
    inference.num_designs=N \
    'contigmap.contigs=[CONTIG]' \
    diffuser.T=50
```

### ProteinMPNN Command Template:

```bash
python colabdesign/rf/designability_test.py \
    --pdb=INPUT.pdb \
    --loc=outputs/DIR \
    --num_seqs=8 \
    --num_recycles=3
```

### Common Contigs:
- `[100]` - 100 residue protein
- `[50:50]` - Two 50-residue chains
- `[40/A10-20/40]` - Scaffold around fixed motif
- `[A:80]` - 80-residue binder to chain A